# Live IBKR Calendar Signals Viewer

View the latest live strategy snapshot saved by `python run_scanner.py --with-history`. This notebook does not connect to IBKR; it only reads `data/results/live_signals`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 180)

p = Path.cwd().resolve()
ROOT = p if (p / 'config.yaml').exists() and (p / 'src').exists() else next(
    (parent for parent in p.parents if (parent / 'config.yaml').exists() and (parent / 'src').exists()),
    p,
)
RESULTS_DIR = ROOT / 'data' / 'results' / 'live_signals'
print('Project root:', ROOT)
print('Live signals dir:', RESULTS_DIR)


## Load Latest Snapshot

In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}

snapshot = read_json(RESULTS_DIR / 'snapshot.json')
live_features = read_csv(RESULTS_DIR / 'live_features.csv')
entry_signals = read_csv(RESULTS_DIR / 'entry_signals.csv')
positions = read_csv(RESULTS_DIR / 'current_positions.csv')
close_signals = read_csv(RESULTS_DIR / 'close_signals.csv')

for df in [live_features, entry_signals]:
    if not df.empty and 'tradeDate' in df.columns:
        df['tradeDate'] = pd.to_datetime(df['tradeDate'], errors='coerce')

print('Snapshot time:', snapshot.get('snapshot_time', 'no snapshot found'))
print('Trade date:   ', snapshot.get('trade_date', ''))
print('Live features:', live_features.shape)
print('Entry signals:', entry_signals.shape)
print('Positions:    ', positions.shape)
print('Close signals:', close_signals.shape)


## Snapshot Summary

In [ ]:
if snapshot:
    display(pd.DataFrame([{'metric': k, 'value': v} for k, v in snapshot.get('counts', {}).items()]))
    display(Markdown('### Universe'))
    print(', '.join(snapshot.get('universe', [])))
else:
    print('No snapshot found. Run: python run_scanner.py --config config.yaml --with-history')


## Close / Hold Recommendations For Current Positions

In [ ]:
if close_signals.empty:
    if positions.empty:
        print('No configured-universe IBKR positions found in the latest snapshot.')
    else:
        print('Positions were found, but no calendar close recommendations were generated.')
else:
    cols = [
        'ticker', 'recommendation', 'close_reason', 'triggered_checks',
        'strike', 'right', 'front_expiry', 'back_expiry', 'front_dte',
        'short_front_qty', 'long_back_qty', 'current_calendar_value',
        'estimated_average_debit', 'estimated_pnl_pct', 'live_spread_zscore',
        'live_spread_pctile', 'live_front_iv', 'live_back_iv', 'note'
    ]
    show = [c for c in cols if c in close_signals.columns]
    display(close_signals[show].sort_values(['recommendation', 'ticker'], ascending=[True, True]).reset_index(drop=True))

    counts = close_signals['recommendation'].value_counts()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(counts.index, counts.values, color=np.where(counts.index == 'CLOSE', '#dc2626', '#059669'))
    ax.set_title('Position Recommendations')
    ax.set_ylabel('Calendars')
    plt.tight_layout()
    plt.show()


## New Entry Signals

In [ ]:
if entry_signals.empty:
    print('No new entry signals in the latest snapshot.')
else:
    cols = [
        'ticker', 'stock_price', 'atm_strike', 'front_expir', 'back_expir',
        'front_dte', 'back_dte', 'front_iv', 'back_iv', 'iv_spread',
        'spread_zscore', 'spread_pctile', 'back_iv_pctile', 'calendar_debit_bs',
        'signal_rank'
    ]
    show = [c for c in cols if c in entry_signals.columns]
    display(entry_signals[show].sort_values('signal_rank', ascending=False, na_position='last').reset_index(drop=True))


## Live Feature Table For All Scanned Tickers

In [ ]:
if live_features.empty:
    print('No live feature rows found.')
else:
    cols = [
        'ticker', 'stock_price', 'atm_strike', 'front_expir', 'back_expir',
        'front_dte', 'back_dte', 'front_iv', 'back_iv', 'iv_spread',
        'spread_zscore', 'spread_pctile', 'back_iv_pctile', 'calendar_debit_bs'
    ]
    show = [c for c in cols if c in live_features.columns]
    display(live_features[show].sort_values('iv_spread', ascending=False).reset_index(drop=True))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    lf = live_features.copy()
    axes[0].bar(lf['ticker'], lf['iv_spread'], color='#2563eb')
    axes[0].axhline(0, color='black', linewidth=1)
    axes[0].set_title('Live IV Spread By Ticker')
    axes[0].tick_params(axis='x', rotation=45)
    if 'spread_zscore' in lf.columns:
        axes[1].bar(lf['ticker'], lf['spread_zscore'], color='#7c3aed')
        axes[1].axhline(0, color='black', linewidth=1)
        axes[1].set_title('Live Spread Z-Score By Ticker')
        axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()


## Raw Current IBKR Positions

In [ ]:
if positions.empty:
    print('No configured-universe positions in latest snapshot.')
else:
    display(positions.sort_values(['ticker', 'expiry', 'strike', 'right']).reset_index(drop=True))


## Snapshot Archive

In [ ]:
archive_dir = RESULTS_DIR / 'archive'
if not archive_dir.exists():
    print('No archive directory found yet.')
else:
    archive = sorted(archive_dir.glob('snapshot_*.json'), reverse=True)
    rows = []
    for path in archive[:30]:
        try:
            snap = json.loads(path.read_text(encoding='utf-8'))
            rows.append({
                'snapshot_time': snap.get('snapshot_time'),
                'entry_signal_rows': snap.get('counts', {}).get('entry_signal_rows'),
                'position_rows': snap.get('counts', {}).get('position_rows'),
                'close_recommendations': snap.get('counts', {}).get('close_recommendations'),
                'file': path.name,
            })
        except Exception:
            pass
    display(pd.DataFrame(rows))


## How To Refresh

In [ ]:
print('Run this from the project folder while TWS/Gateway is open:')
print('python run_scanner.py --config config.yaml --with-history')
print()
print('Then rerun this notebook from the Load Latest Snapshot cell downward.')
